In [ ]:
// Function to get student email and name based on roll number
function GetStudentEmail() {
  var ss = SpreadsheetApp.getActiveSpreadsheet();
  var dopsSheet = ss.getSheetByName("DOPS");
  var residentSheet = ss.getSheetByName("Residents");

  var lastRow = dopsSheet.getLastRow();
  var rollNumber = dopsSheet.getRange(lastRow, 3).getValue();

  var residentData = residentSheet.getDataRange().getValues();
  for (var i = 1; i < residentData.length; i++) {
    if (residentData[i][0] == rollNumber) {
      var studentName = residentData[i][1];
      var studentEmail = residentData[i][2];
      return { studentName: studentName, studentEmail: studentEmail };
    }
  }
  return { studentName: "", studentEmail: "" };
}

// Function to calculate total marks, full marks, and percentage
function calculate() {
  var ss = SpreadsheetApp.getActiveSpreadsheet();
  var dopsSheet = ss.getSheetByName("DOPS");
  var lastRow = dopsSheet.getLastRow();
  var data = dopsSheet.getRange(lastRow, 10, 1, 8).getValues()[0];

  var totalMark = 0;
  var zeroCount = 0;

  for (var i = 0; i < data.length; i++) {
    var val = Number(data[i]);
    if (val === 0) zeroCount++;
    totalMark += val;
  }

  var fullMark = 89 - (zeroCount * 9);
  var percentage = (totalMark / fullMark) * 100;

  return { totalMark: totalMark, fullMark: fullMark, percentage: percentage };
}

// Function to send email to student
function SendEmailStudent() {
  var ss = SpreadsheetApp.getActiveSpreadsheet();
  var dopsSheet = ss.getSheetByName("DOPS");
  var lastRow = dopsSheet.getLastRow();
  var studentInfo = GetStudentEmail();
  var calc = calculate();

  var values = dopsSheet.getRange(lastRow, 2, 1, 20).getValues()[0];
  var body = `Evaluating Faculty: ${values[0]}
Position of Faculty: ${values[20]}
Resident Name: ${studentInfo.studentName}
Resident Roll Number: ${values[1]}
Year: ${values[2]}
Date of Assessment: ${values[3]}
Speciality: ${values[4]}
EPA Topic: ${values[5]}
EPA Number: ${values[6]}
Procedure Name: ${values[7]}
Obtains informed consent: ${values[8]}
Demonstrates appropriate preparation pre procedure: ${values[9]}
Technical ability: ${values[10]}
Aseptic Technique: ${values[11]}
Seeks help where appropriate: ${values[12]}
Post procedural management: ${values[13]}
Communication skill: ${values[14]}
Consideration of patient professionalism: ${values[15]}
Obtained Marks: ${calc.totalMark}
Obtained Percentage: ${calc.percentage.toFixed(2)}%
Overall ability to perform procedure: ${values[16]}
Areas done well: ${values[17]}
Specific action that needs to be taken to achieve desired competency: ${values[18]}

Thank you
Dean Office`;

  var subject = `DOPS; Year ${values[2]}, EPA ${values[5]}, EpA no ${values[6]}`;
  MailApp.sendEmail(studentInfo.studentEmail, subject, body);
}

// Function to get Program Director's name and email
function ProgramDirector() {
  var ss = SpreadsheetApp.getActiveSpreadsheet();
  var dopsSheet = ss.getSheetByName("DOPS");
  var pdSheet = ss.getSheetByName("ProgramDirector");
  var lastRow = dopsSheet.getLastRow();
  var speciality = dopsSheet.getRange(lastRow, 6).getValue();
  var pdData = pdSheet.getDataRange().getValues();

  for (var i = 1; i < pdData.length; i++) {
    if (pdData[i][0] == speciality) {
      return { programDirectorName: pdData[i][1], programDirectorEmail: pdData[i][2] };
    }
  }
  return { programDirectorName: "", programDirectorEmail: "" };
}

// Function to assess faculty performance
function FacultyPerformance() {
  var ss = SpreadsheetApp.getActiveSpreadsheet();
  var dopsSheet = ss.getSheetByName("DOPS");
  var lastRow = dopsSheet.getLastRow();
  var data = dopsSheet.getRange(lastRow, 10, 1, 8).getValues()[0];
  var calc = calculate();

  var mean = data.reduce((a, b) => a + b, 0) / data.length;
  var varianceSum = data.reduce((a, b) => a + Math.pow(b - mean, 2), 0);
  var stanDev = Math.sqrt(varianceSum / data.length);
  var variance = stanDev / mean;

  var discrimination = variance === 0 ? "Poor" : (variance < 0.5 ? "Good" : "Moderate");

  var areaWell = dopsSheet.getRange(lastRow, 19).getValue();
  var areaImp = dopsSheet.getRange(lastRow, 20).getValue();

  var cleanText = text => text.replace(/[.,\/#!$%\^&\*;:{}=\-_`~()]/g, '').split(/\s+/).filter(w => w.length >= 4);
  var lenAreaImp = cleanText(areaImp).length;

  var narrativeSkill = lenAreaImp <= 5 ? "Needs Improvement" : (lenAreaImp <= 10 ? "Can be better" : "Is Great");

  return {
    mean: mean,
    stanDev: stanDev,
    variance: variance,
    discrimination: discrimination,
    narrativeSkill: narrativeSkill
  };
}

// Function to place calculated values in the last row
function Place() {
  var ss = SpreadsheetApp.getActiveSpreadsheet();
  var dopsSheet = ss.getSheetByName("DOPS");
  var lastRow = dopsSheet.getLastRow();
  var calc = calculate();
  var perf = FacultyPerformance();

  dopsSheet.getRange(lastRow, 26).setValue(calc.fullMark);
  dopsSheet.getRange(lastRow, 27).setValue(calc.totalMark);
  dopsSheet.getRange(lastRow, 28).setValue(calc.percentage);
  dopsSheet.getRange(lastRow, 29).setValue(perf.mean);
  dopsSheet.getRange(lastRow, 30).setValue(perf.stanDev);
  dopsSheet.getRange(lastRow, 31).setValue(perf.variance);
  dopsSheet.getRange(lastRow, 32).setValue(perf.discrimination);
  dopsSheet.getRange(lastRow, 33).setValue(perf.narrativeSkill);
}

// Function to send Program Director email (and also to DOPS sheet column B)
function SendEmailProgramDirector() {
  var ss = SpreadsheetApp.getActiveSpreadsheet();
  var dopsSheet = ss.getSheetByName("DOPS");
  var lastRow = dopsSheet.getLastRow();
  var studentInfo = GetStudentEmail();
  var calc = calculate();
  var pd = ProgramDirector();
  var perf = FacultyPerformance();

  var values = dopsSheet.getRange(lastRow, 2, 1, 20).getValues()[0];
  var ccEmail = dopsSheet.getRange(lastRow, 2).getValue();  // Column B

  var body = `Program Director: ${pd.programDirectorName}
Evaluating Faculty: ${values[0]}
Position of Faculty: ${values[20]}
Resident Name: ${studentInfo.studentName}
Resident Roll Number: ${values[1]}
Year: ${values[2]}
Speciality: ${values[4]}
EPA Topic: ${values[5]}
EPA Number: ${values[6]}

Student Evaluation
Procedure Name: ${values[7]}
Obtains informed consent: ${values[8]}
Demonstrates appropriate preparation pre procedure: ${values[9]}
Technical ability: ${values[10]}
Aseptic Technique: ${values[11]}
Seeks help where appropriate: ${values[12]}
Post procedural management: ${values[13]}
Communication skill: ${values[14]}
Consideration of patient professionalism: ${values[15]}
Obtained Marks: ${calc.totalMark}
Obtained Percentage: ${calc.percentage.toFixed(2)}%
Overall ability to perform procedure: ${values[16]}
Areas done well: ${values[17]}
Specific action that needs to be taken to achieve desired competency: ${values[18]}

Faculty Performance
(This is based on scoring pattern and may not reflect truly, however it is helpful for program director to identify extreme pattern repeatedly.)

The ability of faculty for identifying skills across various items seems to be ${perf.discrimination}.
The skill of faculty in narrative writeup: ${perf.narrativeSkill}.

Thank you
Dean Office`;

  var subject = `DOPS; Year ${values[2]}, EPA ${values[5]}, EpA no ${values[6]}`;
  MailApp.sendEmail({
    to: pd.programDirectorEmail,
    cc: ccEmail,
    subject: subject,
    body: body
  });
}
